# ⚡ 100% 合規雲端 GPU 2K 寫實私房寫真與「換裝動態」圖生影工作站 (`auto_worker.ipynb`)

- **解決 SDXL 77-Token 截斷問題**：使用 Dual-CLIP (`prompt` 傳入濃縮核心人設 + `prompt_2` 傳入完整細節)，確保「20歲法越混血、冷白真實毛孔瓷肌、165cm 50kg 自然 F 罩杯、極薄透明網紗蕾絲內衣、和室全身鏡/純白床單」100% 生效，絕不跑偏成街頭日常服裝。
- **真 AI 圖生影片（根據產製圖片生成「換衣服/整理薄紗內衣」真動作）**：直接讀取本批次剛產出的寫真照片 (`portrait_1.jpg`, `portrait_2.jpg`)，透過 `Lightricks/LTX-Video` 賦予人物緩緩褪下薄袍、拉動蕾絲肩帶、轉身展現曲線的真實肢體動態（非單純鏡頭搖晃）。

In [ ]:
import os, sys, json, time, shutil, subprocess, gc

print("🚀 [1/5] 正在安裝最新版 Diffusers 與加速套件...")
subprocess.run("pip install -q -U diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg ftfy", shell=True, check=True)

print("🔄 [2/5] 正在從 GitHub 同步最新任務設定與隨機 Prompt 引擎...")
if not os.path.exists("colab-ai-media-lab"):
    subprocess.run("git clone https://github.com/Chenponpon/colab-ai-media-lab.git", shell=True, check=True)
else:
    subprocess.run("cd colab-ai-media-lab && git fetch --all && git reset --hard origin/main", shell=True, check=True)

sys.path.insert(0, os.path.abspath("colab-ai-media-lab/scripts"))
from prompt_randomizer import generate_diversified_prompt, DEFAULT_NEGATIVE_PROMPT

with open("colab-ai-media-lab/queue/current_job.json", "r", encoding="utf-8") as f:
    job = json.load(f)

import torch, psutil
from PIL import Image, ImageFilter
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

image_times = []
image_peak_vram = 0.0
max_res_str = "1536x2304"
generated_portraits = []
generated_recipes = []
human_ts = time.strftime("%Y-%m-%d %H:%M:%S")

# ==================== 階段一：RealVisXL V5.0 雙向 Prompt 無截斷 2K 寫實生圖 ====================
img_cfg = job.get("image_generation", {})
if img_cfg.get("enabled", True):
    print("🎨 [3/5] 正在載入 RealVisXL V5.0 寫實人像模型 (啟用 Dual-CLIP 防截斷 + 正負面雙 Prompt)...")
    from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
    model_id = img_cfg.get("model_id", "SG161222/RealVisXL_V5.0")
    pipe = StableDiffusionXLPipeline.from_pretrained(model_id, torch_dtype=torch.float16, variant="fp16").to("cuda")
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++")
    pipe.vae.enable_tiling()

    num_imgs = img_cfg.get("num_images", 4)
    for i in range(num_imgs):
        nat_w, nat_h = (1024, 1536) if i == num_imgs - 1 else (img_cfg.get("native_width", 960), img_cfg.get("native_height", 1440))
        out_w, out_h = (1536, 2304) if i == num_imgs - 1 else (img_cfg.get("target_width", 1440), img_cfg.get("target_height", 2160))
        full_prompt, recipe = generate_diversified_prompt()

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        gen = torch.Generator("cuda").manual_seed(recipe["seed"])
        img = pipe(
            prompt=recipe["clip_short_prompt"],
            prompt_2=full_prompt,
            negative_prompt=recipe.get("negative_prompt", DEFAULT_NEGATIVE_PROMPT),
            negative_prompt_2=recipe.get("negative_prompt", DEFAULT_NEGATIVE_PROMPT),
            width=nat_w, height=nat_h,
            num_inference_steps=img_cfg.get("steps", 30),
            guidance_scale=img_cfg.get("guidance_scale", 5.5),
            generator=gen
        ).images[0]

        if img.size != (out_w, out_h):
            img = img.resize((out_w, out_h), Image.LANCZOS)
        img = img.filter(ImageFilter.UnsharpMask(radius=1.0, percent=45, threshold=3))

        dt = round(time.time() - t0, 2)
        pk = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
        image_times.append(dt)
        image_peak_vram = max(image_peak_vram, pk)
        max_res_str = f"{out_w}x{out_h}"

        recipe["timestamp"] = human_ts
        recipe["model_engine"] = "RealVisXL V5.0 (SDXL Dual-CLIP 30-Steps)"
        recipe["elapsed_sec"] = dt
        recipe["peak_vram_gb"] = pk
        recipe["resolution"] = f"{out_w}x{out_h}"

        fname = f"portrait_{i+1}.jpg"
        rname = f"portrait_{i+1}_recipe.json"
        img.save(fname, quality=96)
        with open(rname, "w", encoding="utf-8") as rf:
            json.dump(recipe, rf, ensure_ascii=False, indent=2)

        generated_portraits.append(fname)
        generated_recipes.append(recipe)
        print(f"\n================ 🎲 第 {i+1} 張 [{out_w}x{out_h}] 耗時: {dt}s | 峰值 VRAM: {pk} GB ================")
        print(recipe["summary_text"])

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

# ==================== 階段二：根據產製圖片生成「換衣服/調整薄紗內衣」真實人物動態影片 (I2V) ====================
vid_cfg = job.get("video_generation", {})
video_times = []
video_peak_vram = 0.0
video_model_used = "Lightricks/LTX-Video (Real Human Changing-Clothes Motion I2V)"

if vid_cfg.get("enabled", True) and generated_portraits:
    print("\n🎬 [4/5] 正在載入圖生影引擎 (LTX-Video I2V) 將剛產出的寫真照片轉為『換裝/整理薄紗內衣』真實人物動態影片...")
    from diffusers import LTXImageToVideoPipeline
    from diffusers.utils import export_to_video

    v_pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
    v_pipe.enable_model_cpu_offload()
    v_pipe.vae.enable_tiling()

    num_vids = min(vid_cfg.get("num_videos_from_generated", 2), len(generated_portraits))
    vw, vh = vid_cfg.get("width", 512), vid_cfg.get("height", 768)
    for idx in range(num_vids):
        img_p = generated_portraits[idx]
        rcp = generated_recipes[idx]
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        in_img = Image.open(img_p).convert("RGB").resize((vw, vh), Image.LANCZOS)
        frames = v_pipe(
            image=in_img,
            prompt=rcp["video_motion_prompt"],
            negative_prompt=rcp["video_negative_prompt"],
            height=vh, width=vw,
            num_frames=vid_cfg.get("num_frames", 81),
            guidance_scale=vid_cfg.get("guidance_scale", 5.0),
            num_inference_steps=vid_cfg.get("steps", 25)
        ).frames[0]
        dt = round(time.time() - t0, 2)
        pk = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
        video_times.append(dt)
        video_peak_vram = max(video_peak_vram, pk)

        vname = f"video_{idx+1}_from_portrait_{idx+1}.mp4"
        export_to_video(frames, vname, fps=vid_cfg.get("fps", 16))
        print(f"✅ 換裝動態影片完成: {vname} ({vw}x{vh}, 81影格/5秒) | 動作: {rcp['video_motion_zh']} | 耗時: {dt}s | 峰值 VRAM: {pk} GB")

# ==================== 階段三：輸出硬體壓測總結報告 ====================
bench_report = {
    "gpu_name": gpu_name,
    "vram_gb": vram_gb,
    "ram_gb": ram_gb,
    "avg_image_time_sec": round(sum(image_times)/len(image_times), 2) if image_times else 0,
    "image_peak_vram_gb": image_peak_vram,
    "max_resolution_tested": max_res_str,
    "avg_video_time_sec": round(sum(video_times)/len(video_times), 2) if video_times else 0,
    "video_peak_vram_gb": video_peak_vram,
    "video_model": video_model_used
}
with open("benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench_report, bf, ensure_ascii=False, indent=2)

print("\n🏆 [5/5] 雲端 2K 寫實生圖與換裝圖生影全部完成！")